# Edge Filters

##### Author: [Radoslav Neychev](https://www.linkedin.com/in/radoslav-neychev/), telegram: [@rads_ai](https://t.me/rads_ai)

In [1]:
# do not change the code in the block below
# __________start of block__________
import json
import os
import cv2
import random

import numpy as np
import torch
import torchvision
from IPython.display import clear_output
from matplotlib import pyplot as plt
from torchvision.datasets import FashionMNIST

# __________end of block__________

Let us continue working with the [FashionMNIST](https://github.com/zalandoresearch/fashion-mnist) dataset.

__Your task is to implement an edge detection mechanism (Sobel filter) and a simplified version of the histogram of oriented gradients.__

Classification accuracy will not be evaluated; you only need to implement the functions and submit them to the contest.

The notebook contains several tests that will help you debug your solution.

In [ ]:
# do not change the code in the block below
# __________start of block__________

train_fmnist_data = FashionMNIST(
    ".", train=True, transform=torchvision.transforms.ToTensor(), download=True
)

train_data_loader = torch.utils.data.DataLoader(
    train_fmnist_data, batch_size=32, shuffle=True, num_workers=2
)

random_batch = next(iter(train_data_loader))
_image, _label = random_batch[0][0], random_batch[1][0]
plt.figure()
plt.imshow(_image.reshape(28, 28))
plt.title(f"Image label: {_label}")
# __________end of block__________

#### Step 1. Sobel filtering
Implement the function `compute_sobel_gradients_two_loops`. Part of the function is already written; please do not change the existing code.

In [3]:
# do not change the code in the block below
# __________start of block__________
import numpy as np
def compute_sobel_gradients_two_loops(image):
    # Get image dimensions
    height, width = image.shape

    # Initialize output gradients
    gradient_x = np.zeros_like(image, dtype=np.float64)
    gradient_y = np.zeros_like(image, dtype=np.float64)

    # Pad the image with zeros to handle borders
    padded_image = np.pad(image, ((1, 1), (1, 1)), mode='constant', constant_values=0)
# __________end of block__________

    # Define the Sobel kernels for X and Y gradients
    sobel_x = None # YOUR CODE HERE
    sobel_y = None # YOUR CODE HERE

    # Apply Sobel filter for X and Y gradients using convolution
    for i in range(1, height + 1):
        for j in range(1, width + 1):
            pass
            # YOUR CODE HERE
    return gradient_x, gradient_y

To check the code you wrote, we will use the already implemented version from OpenCV. Because the padding operation may be performed differently, we will ignore differences at the image borders.

In [4]:
# do not change the code in the block below
# __________start of block__________
def compute_sobel_gradients_opencv(image):
    # Apply Sobel filter for horizontal and vertical gradients
    sobel_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    
    # Return gradients in both directions
    return sobel_x, sobel_y
# __________end of block__________


In [ ]:
# do not change the code in the block below
# __________start of block__________
image = train_fmnist_data[7][0][0].numpy()
gradients_two_loops = compute_sobel_gradients_two_loops(image)
gradients_opencv = compute_sobel_gradients_opencv(image)

assert np.allclose(gradients_two_loops[0][1:-1, 1:-1], gradients_opencv[0][1:-1, 1:-1], atol=1e-2), "gradients_two_loops[0] and gradients_opencv[0] are not close"
assert np.allclose(gradients_two_loops[1][1:-1, 1:-1], gradients_opencv[1][1:-1, 1:-1], atol=1e-2), "gradients_two_loops[1] and gradients_opencv[1] are not close"
print("Everything seems fine!")
# __________end of block__________


In [ ]:
image = random.choice(train_fmnist_data)[0][0].numpy()
gradients_two_loops = compute_sobel_gradients_two_loops(image)

plt.subplot(1, 3, 1)
plt.imshow(image)
plt.subplot(1, 3, 2)
plt.imshow(gradients_two_loops[0])
plt.subplot(1, 3, 3)
plt.imshow(gradients_two_loops[1])

#### Step 2. Computing gradients in polar coordinates.
Implement two functions:
 * `compute_gradient_magnitude`, which computes the Euclidean norm of the gradient
 * `compute_gradient_direction`, which computes its direction as an angle relative to the $x$ axis. The returned angle must be in the range $(-180; 180]$.

In [6]:
import numpy as np # for your convenience when you copy the code to the contest
def compute_gradient_magnitude(sobel_x, sobel_y):
    '''
    Compute the magnitude of the gradient given the x and y gradients.

    Inputs:
        sobel_x: numpy array of the x gradient.
        sobel_y: numpy array of the y gradient.

    Returns:
        magnitude: numpy array of the same shape as the input [0] with the magnitude of the gradient.
    '''
    # YOUR CODE HERE
    return


def compute_gradient_direction(sobel_x, sobel_y):
    '''
    Compute the direction of the gradient given the x and y gradients. Angle must be in degrees in the range (-180; 180].
    Use arctan2 function to compute the angle.

    Inputs:
        sobel_x: numpy array of the x gradient.
        sobel_y: numpy array of the y gradient.

    Returns:
        gradient_direction: numpy array of the same shape as the input [0] with the direction of the gradient.
    '''
    # YOUR CODE HERE
    return


Small tests for the `compute_gradient_direction` function

In [ ]:
# do not change the code in the block below
# __________start of block__________
image = train_fmnist_data[7][0][0].numpy()
gradients_two_loops = compute_sobel_gradients_two_loops(image)

magnitudes = compute_gradient_magnitude(gradients_two_loops[0], gradients_two_loops[1])
angles = compute_gradient_direction(gradients_two_loops[0], gradients_two_loops[1])
assert np.all(magnitudes >= 0), "Magnitudes should be non-negative"
assert np.all(angles > -180) and np.all(angles <= 180), "Angles should be in the range (-180, 180]"
print("Everything seems fine!")
# __________end of block__________


Example visualization of the resulting edges obtained using the Sobel filter:

In [ ]:
image = random.choice(train_fmnist_data)[0][0].numpy()
magnitudes = compute_gradient_magnitude(*compute_sobel_gradients_two_loops(image))

plt.subplot(1, 2, 1)
plt.imshow(image)
plt.subplot(1, 2, 2)
plt.imshow(magnitudes)

#### Step 3. Simplified HoG variant.
You need to implement the histogram of oriented gradients (HoG). In general, this is done as follows:
1. Convert the image to a single channel. If the image is color, it should be converted to grayscale. You can use the formula from [Wiki](https://en.wikipedia.org/wiki/Grayscale):
$$
\text{brightness}_{i, j} = \text{Red}_{i, j} * 0.2126 + \text{Green}_{i, j} * 0.7152 + \text{Blue}_{i, j} * 0.0722,
$$
but for simplicity we will just average all channels.

*Note: this is far from the only conversion method, and it may be nonlinear. Details are available at the link above*.

2. Compute image gradients using the Sobel filter.

3. Determine the gradient direction and norm for each pixel.

4. Build histograms of gradient directions. To do this, the image is split into non-overlapping square cells. The size of each cell is set by the `pixels_per_cell` parameter, which by default is equal to `(cell_size, cell_size)`; the `cell_size` parameter is defined by default below in the code. For each image cell, a histogram of gradient directions is built. To do this:
    * Each gradient direction is assigned to a particular bin (9 bins total from -180 to 180).
    * For each bin, all norms that fall into it are summed.
    * The constructed histogram is normalized (so that the sum of all bins equals 1).
    * *Note: we recommend using `np.histogram` to obtain results where edge values match. Since the elements are positive, normalization is performed by their total sum if the histogram is not empty.*
5. Block normalization of histograms. For greater adaptability of the method to illumination changes, normalization is performed. You can read more about it [here](https://scikit-image.org/docs/dev/auto_examples/features_detection/plot_hog.html). In this assignment, it **will not** be performed.

At this stage, you only need to complete steps 1-4. Note that **block normalization** of histograms is not performed (this step is skipped).

In [16]:
cell_size = 7
def compute_hog(image, pixels_per_cell=(cell_size, cell_size), bins=9):
    # 1. Convert the image to grayscale if it's not already (assuming the image is in RGB or BGR)
    if len(image.shape) == 3:
        image = np.mean(image, axis=2)  # Simple averaging to convert to grayscale
    
    # 2. Compute gradients with Sobel filter
    gradient_x, gradient_y = # YOUR CODE HERE

    # 3. Compute gradient magnitude and direction
    magnitude = # YOUR CODE HERE
    direction = # YOUR CODE HERE

    # 4. Create histograms of gradient directions for each cell
    cell_height, cell_width = pixels_per_cell
    n_cells_x = image.shape[1] // cell_width
    n_cells_y = image.shape[0] // cell_height

    histograms = np.zeros((n_cells_y, n_cells_x, bins))

    for i in range(n_cells_y):
        for j in range(n_cells_x):
            pass
            # YOUR CODE HERE
    return histograms

To run the tests below, the files `hog_data.npy` and `image_data.npy` from the repository must be available in the same directory as the notebook. You can download them from the repository.

In [ ]:
# do not change the code in the block below
# __________start of block__________
image = random.choice(train_fmnist_data)[0][0].numpy()

hog = compute_hog(image)
assert hog.shape == (4, 4, 9), "hog should have shape (4, 4, 9) for the FashionMNIST image with default parameters"
print("Everything seems fine!")

assert os.path.exists("hog_data.npy") and os.path.exists("image_data.npy"), "hog_data.npy and image_data.npy should be in the same directory as the notebook"
with open("hog_data.npy", "rb") as f:
    hog_data = np.load(f, allow_pickle=True)
with open("image_data.npy", "rb") as f:
    image_data = np.load(f, allow_pickle=True)
for test_image, test_hog in zip(image_data, hog_data):
    hog = compute_hog(test_image)
    assert np.allclose(hog, test_hog), "hog should be the same"

# __________end of block__________


Visualization of the resulting histograms. The grid looks slightly shifted because of the large pixel size; this is normal.

In [ ]:
#plot all the histograms for (3, 3) cells:
image = random.choice(train_fmnist_data)[0][0].numpy()
hog = compute_hog(image)

# draw cells on the image
plt.imshow(image)
for i in range(4):
    for j in range(4):
        plt.gca().add_patch(plt.Rectangle((j * cell_size, i * cell_size), cell_size, cell_size, fill=False, edgecolor='red', linewidth=1))
plt.show()


plt.figure(figsize=(10, 10))
for i in range(4):
    for j in range(4):
        plt.subplot(4, 4, i * 4 + j + 1)
        plt.bar(range(len(hog[i, j])), hog[i, j])
        plt.title(f"Cell {i}, {j}")
plt.show()


### Submitting the assignment
Submit the functions to the task in the contest. Do not forget that when pasting your code, all imported libraries must also be pasted together with your code. You must not use anything other than `numpy` when writing the solution.

This completes the assignment. Congratulations!